### Explorationb of BERT Variants
* Why BERT Variants?
    * While BERT is apowerful, it has limitations like large computational requirements and inefficiencies in capturing certain nuances
    * BERT variants optimize the model for specific tasks, improve performances, or reduce computational overhead
    * Key BERT variants
        * RoBERTa (Robustly Optimized BERT)
            * Removes Next Sentence Prediction tasks for better efficiency
            * Trains on more data with larger batch size
            * Usecase: superior performance in tasks requiring deeper context
        * DistilBERT 
            * A distilled (smaller) version of BERT that retains 97% of BERT's performance while being 60% faster
            * Usecase : Ideal for real-time applications and resource constrained environments
        * ALBERT (A Lite Bert)
            * Reduces memory consumption by factorizing embeddings and sharing parameters across layers
            * Use Case: Suitable for large -scale pre-trained and downstream tasks with memory limitations
        * BERTweet
            * Fine-Tuned on Tweeter Data
            * Use case : Social Media Sentiment Analysis , hashtag predictions

### Transfer Learning in NLP with Transformer Models
* What is Transfer learning?
    * Transfer learning involves pre-training a model on a large dataset and fine -tuning it for sepcific downstream tasks
    * Adv;
        * Reduces the need for task-specific labeled data
        * Speeds up taraining and improves performance on specialized tasks

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset



In [2]:
dataset= load_dataset("SetFit/ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [3]:
# Load RoBERTa tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=4)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
#  Tokenize datase
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenize_datset = dataset.map(tokenize_function, batched= True)



Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [5]:
tokenized_dataset = tokenize_datset.remove_columns(["text"])
tokenize_datset = tokenize_datset.rename_column("label", "labels")

tokenize_datset.set_format("torch")


In [6]:
train_dataset  = tokenize_datset["train"]
test_dataset = tokenize_datset["test"]

In [7]:
training_args = TrainingArguments(
    output_dir="./",
    eval_strategy="epoch",
    learning_rate=2e-15,
    per_device_eval_batch_size=16,
    per_device_train_batch_size=16,
    num_train_epochs= 3,
    weight_decay=0.01,
    save_steps=500
    )

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset = test_dataset,
    processing_class = tokenizer,

)

In [8]:
trainer.train()

ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
# Evaluate Models
results = trainer.evaluate()
print("Evaluation Results", results)

In [ ]:
trainer.create_model_card()